In [1]:
# Save global mortality by year and for each ensemble

In [2]:
import os
import xarray as xr
import numpy as np
import dask.array as da
import warnings
from utils.mortality_utils import att_frac
from utils.mortality_utils import mortality

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"

In [4]:
# === Load data ===
bmr_file = "GBD_BMR_Country_Mask_COPD_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)

In [5]:
n_samples = 1000

# === Scalar distributions (assuming normal dist.) ===
# TMREL from GBD21
tmrel_mean = 32.4
tmrel_std = (35.7 - 29.1) / (2 * 1.96)
tmrel_samples = np.random.normal(tmrel_mean, tmrel_std, size=n_samples)

# Beta from RR per 10ppb
RR_10 = 1.074
RR_10_lower = 1.014
RR_10_upper = 1.137
beta_mean = np.log(RR_10) / 10
beta_std = (np.log(RR_10_upper) - np.log(RR_10_lower)) / (2 * 1.96 * 10)
beta_samples = np.random.normal(beta_mean, beta_std, size=n_samples)

# Convert to Dask for broadcasting
tmrel_dask = da.from_array(tmrel_samples[:, np.newaxis, np.newaxis],
                           chunks=(100, 1, 1))
beta_dask = da.from_array(beta_samples[:, np.newaxis, np.newaxis],
                          chunks=(100, 1, 1))

In [6]:
def monte_carlo(n_samples, BMR, POP, O3):
    lat = BMR["lat"]
    lon = BMR["lon"]

    # === BMR ===
    bmr_m = BMR.sel(quantile="mean")
    bmr_l = BMR.sel(quantile="lower")
    bmr_u = BMR.sel(quantile="upper")

    BMR_mean = bmr_m.chunk({"lat": 180, "lon": 360})
    BMR_lower = bmr_l.chunk({"lat": 180, "lon": 360})
    BMR_upper = bmr_u.chunk({"lat": 180, "lon": 360})

    # Standard deviation for BMR
    bmr_std = (BMR_upper - BMR_lower) / (2 * 1.96)

    # Sample BMR: shape = (samples, lat, lon)
    bmr_samples = da.random.normal(
        loc=BMR_mean.data, scale=bmr_std.data,
        size=(n_samples, len(lat), len(lon)),
        chunks=(100, 180, 360))

    POP = POP.chunk({"lat": 180, "lon": 360})
    O3 = O3.chunk({"lat": 180, "lon": 360})

    # Broadcast POP and x to sample dimension
    pop_samples = da.broadcast_to(POP.data, (n_samples, len(lat), len(lon)))
    O3_samples = da.broadcast_to(O3.data, (n_samples, len(lat), len(lon)))

    return bmr_samples, pop_samples, O3_samples

In [7]:
def create_stats(da):
    mean = da.mean()
    lower = da.quantile(0.025, dim="sample")
    upper = da.quantile(0.975, dim="sample")

    combined = xr.DataArray(
        data=[mean.values, lower.values, upper.values],
        coords={"quantile": ["mean", "0.025", "0.975"]},
        dims=["quantile"],
        name="Mortality"
    )
    return combined

In [ ]:
warnings.filterwarnings('ignore')
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/mortality/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(7, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        o3_file = f"OSDMA8_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        o3_path = os.path.join(O3_DIR, o3_file)
        o3 = xr.open_dataarray(o3_path)

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-7 km distance
        o3 = o3.reindex_like(BMR, method="nearest", tolerance=1e-9, fill_value=0)
        population = population.reindex_like(BMR, method="nearest", tolerance=1e-9)

        # Flag if any nans present (i.e. reindex was out of tolerance distance)
        assert not population.isnull().any()

        M = []

        for year in o3["year"].values:
            print(f"Processing year {year}")
            POP = population.sel(year=year)
            O3 = o3.sel(year=year)
            BMR_samples, POP_samples, O3_samples = monte_carlo(n_samples, BMR,
                                                               POP, O3)

            AF = att_frac(O3_samples, tmrel_dask, beta_dask)
            mortality_samples = mortality(AF, BMR_samples, POP_samples)
            mortality_year = xr.DataArray(
                mortality_samples,
                dims=("sample", "lat", "lon"),
                coords={"sample": np.arange(n_samples),
                        "lat": BMR.lat, "lon": BMR.lon},
                name="Mortality"
            ).chunk({"sample": -1, "lat": -1, "lon": -1})
            del mortality_samples, AF, BMR_samples, POP_samples, O3_samples
            mortality_year = mortality_year.persist()

            print("Global sum processing")
            globalsum = mortality_year.sum(dim=("lat", "lon")).compute()
            globalsum_stats = create_stats(globalsum)
            M.append(globalsum_stats)
            del globalsum, mortality_year

            mean_mortality = globalsum_stats.sel(quantile="mean").round().values
            print(f"Mean mortality rate for {year} is {mean_mortality}")

        mortality_timeseries = xr.concat(
            M,
            dim=(xr.DataArray(o3["year"].values,
                              dims="year", name="year"))
        )

        out_file = f"Mortality_global_sum_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving mortality timeseries to {out_path}")
        description = ("Global mean [95% CI] COPD mortality due to surface ozone - scripts "
                       "by A.F. Wells (2025)")
        mortality_timeseries.attrs["description"] = description
        mortality_timeseries.attrs["ensemble_number"] = ens_num
        mortality_timeseries.attrs["scenario"] = scenario
        mortality_timeseries.to_netcdf(out_path)

Processing ARISE, Ensemble 07
Processing year 2035
Global sum processing
Mean mortality rate for 2035 is 501917.0
Processing year 2036
Global sum processing
Mean mortality rate for 2036 is 489510.0
Processing year 2037
Global sum processing
Mean mortality rate for 2037 is 505638.0
Processing year 2038
Global sum processing
Mean mortality rate for 2038 is 519529.0
Processing year 2039
Global sum processing
Mean mortality rate for 2039 is 513835.0
Processing year 2040
Global sum processing
Mean mortality rate for 2040 is 515003.0
Processing year 2041
Global sum processing
Mean mortality rate for 2041 is 519935.0
Processing year 2042
Global sum processing
Mean mortality rate for 2042 is 497987.0
Processing year 2043
Global sum processing
Mean mortality rate for 2043 is 507436.0
Processing year 2044
Global sum processing
Mean mortality rate for 2044 is 500846.0
Processing year 2045
Global sum processing
Mean mortality rate for 2045 is 514484.0
Processing year 2046
Global sum processing
Mea

In [9]:
from pympler import asizeof

# Collect all global variables and compute their total size
sizes = []
for name, var in list(globals().items()):
    if name.startswith("__") and name.endswith("__"):
        continue  # Skip built-in entries like __name__, __file__, etc.
    try:
        size = asizeof.asizeof(var)
        sizes.append((name, size))
    except Exception as e:
        print(f"Could not measure size of {name}: {e}")

# Sort and print top 10 variables by size
print("\nTop memory-using global variables:")
for name, size in sorted(sizes, key=lambda x: x[1], reverse=True)[:10]:
    print(f"{name}: {size / 1024 / 1024:.2f} MB")


Could not measure size of exit: invalid option: reset(base=-14288)
Could not measure size of quit: invalid option: reset(base=-14288)
Could not measure size of sys: invalid option: reset(base=-14288)
Could not measure size of BMR: invalid option: reset(base=-14288)
Could not measure size of mortality_year: 'dtype'

Top memory-using global variables:
population: 9986.80 MB
POP: 5042.94 MB
O3: 1730.56 MB
o3: 1681.13 MB
np: 0.70 MB
da: 0.24 MB
os: 0.14 MB
warnings: 0.03 MB
xr: 0.03 MB
tmrel_dask: 0.02 MB
